## Parcellated ISC, RSA & Brain-Behavior Analysis

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import importlib

from yy_fmri_kit.event_isc.extraction import parcel
importlib.reload(parcel)
from yy_fmri_kit.event_isc.extraction.parcel import (
    Config,
    load_events,
    load_timeseries,
    extract_post_patterns,
    compute_isc,
    compute_rsa,
    compute_rsa_multivariate,
    permutation_test,
    load_affiliation,
    make_behavioral_rdm,
    compute_brain_behavior_rsa,
    permutation_test_brain_behavior,
    fdr_correct,
    results_to_dataframe,
)
from yy_fmri_kit.visualization import pattern_analysis
importlib.reload(pattern_analysis)
from yy_fmri_kit.visualization.pattern_analysis import (
    plot_rdm,
    plot_rdm_comparison,
    plot_group_rdm,
    plot_condition_bar,
    plot_isc_parcels,
    plot_network_summary,
    plot_null_distribution,
    plot_subject_isc,
    plot_rsa_scatter,
    plot_similarity_matrices,
    plot_brain_behavior_scatter,
    plot_brain_behavior_bar,
)

## 1. Configuration & Data Loading

In [ ]:
cfg = Config(
    data_dir     = Path('/path/to/data/derivatives/parcellated'),
    events_csv   = Path('/path/to/behavioral_analyses/data/combined_events_with_bids.csv'),
    subjects = [f'sub-{i}' for i in range(1, 34)],
    tr           = 1.0,
    shift_tr     = 4,
    run_types    = ['AntiLeft', 'AntiRight', 'ProLeft', 'ProRight'],  # ← missing
    subject_col  = 'bids_id',
    run_col      = 'run',
    post_col     = 'post_id',
    onset_col    = 'onset_s',
    duration_col = 'duration_s',
    tsv_glob     = '{subject}/{subject}_*_task-{run_type}_*atlas-Schaefer2018*timeseries.tsv',
    n_perms      = 1000,
    fdr_q        = 0.05,
)

events_df  = load_events(cfg)
ts_dict    = load_timeseries(cfg)
patterns   = extract_post_patterns(ts_dict, events_df, cfg)

# Parcel names — same order as axis-1 in all pattern arrays
parcel_names = ts_dict[(cfg.subjects[0], cfg.run_types[0])].columns.tolist()
print(f'\n{len(parcel_names)} parcels')

In [ ]:
# show event.df types and counts
print("\nEvent DataFrame info:")
print(events_df.info())
print("\nEvent DataFrame head:")
print(events_df.head())

## 2. ISC — Inter-Subject Correlation

In [ ]:
all_isc    = {}
isc_nulls  = {}

for run_type in cfg.run_types:
    print(f'\n--- {run_type} ---')
    obs, p_vals, null   = permutation_test(patterns[run_type], cfg, analysis='isc')
    rejected, p_fdr     = fdr_correct(p_vals, q=cfg.fdr_q)
    _, isc_subj         = compute_isc(patterns[run_type])

    all_isc[run_type]   = results_to_dataframe(
        parcel_names, obs, p_vals, rejected, p_fdr,
        subj_vals=isc_subj, subjects=cfg.subjects
    )
    isc_nulls[run_type] = (obs, p_vals, null, isc_subj)
    print(f'  {rejected.sum()} / {len(parcel_names)} parcels significant | '
          f'mean r = {obs.mean():.3f} | max r = {obs.max():.3f}')

In [ ]:
# Summary table
pd.DataFrame({
    rt: {'n_sig': df['significant'].sum(),
         'mean_r': df['r'].mean().round(3),
         'max_r':  df['r'].max().round(3)}
    for rt, df in all_isc.items()
}).T

### ISC Visualisations

In [ ]:
# Mean ISC ± SEM across parcels, one bar per condition
fig = plot_condition_bar(all_isc, title='ISC across conditions')
plt.show()

In [ ]:
# Top 20 significant parcels for each condition
for run_type in cfg.run_types:
    fig = plot_isc_parcels(all_isc[run_type], run_type=run_type, top_n=20)
    plt.show()

In [ ]:
# ISC aggregated by Schaefer 7-network
fig = plot_network_summary(all_isc, title='ISC by network')
plt.show()

In [ ]:
# Per-subject ISC in a parcel of interest (e.g. Default PCC)
# Find a parcel name with 'Default' and 'PCC'
pcc_parcels = [p for p in parcel_names if 'Default' in p and 'PCC' in p]
print('PCC parcels:', pcc_parcels[:5])

pcc_idx = parcel_names.index(pcc_parcels[0])

for run_type in cfg.run_types:
    obs, p_vals, null, isc_subj = isc_nulls[run_type]
    fig = plot_subject_isc(isc_subj, pcc_idx, cfg.subjects,
                            parcel_name=pcc_parcels[0], run_type=run_type)
    plt.show()

In [ ]:
# Null distribution for one parcel — diagnostic plot
run_type = 'AntiLeft'
obs, p_vals, null, _ = isc_nulls[run_type]
fig = plot_null_distribution(null, obs, pcc_idx,
                              parcel_name=pcc_parcels[0],
                              run_type=run_type,
                              p_val=p_vals[pcc_idx])
plt.show()

## 3. RSA — Representational Similarity Analysis (post × post)

In [ ]:
all_rsa   = {}
rsa_nulls = {}

for run_type in cfg.run_types:
    print(f'\n--- {run_type} ---')
    obs, p_vals, null = permutation_test(patterns[run_type], cfg, analysis='rsa')
    rejected, p_fdr   = fdr_correct(p_vals, q=cfg.fdr_q)
    _, rsa_subj       = compute_rsa(patterns[run_type])

    all_rsa[run_type]   = results_to_dataframe(
        parcel_names, obs, p_vals, rejected, p_fdr,
        subj_vals=rsa_subj, subjects=cfg.subjects
    )
    rsa_nulls[run_type] = (obs, p_vals, null, rsa_subj)
    print(f'  {rejected.sum()} / {len(parcel_names)} parcels significant | '
          f'mean r = {obs.mean():.3f}')

### RSA Visualisations

In [ ]:
# Group-average RDMs for all conditions
fig = plot_group_rdm(patterns)
plt.show()

In [ ]:
# One subject's RDMs across all conditions
fig = plot_rdm_comparison(patterns, subject_idx=0,
                           subject_label=cfg.subjects[0])
plt.show()

In [ ]:
# RSA scatter for one subject in one condition (RDM correlation)
fig = plot_rsa_scatter(patterns['AntiLeft'], subject_idx=0,
                        subject_label=cfg.subjects[0], run_type='AntiLeft')
plt.show()

In [ ]:
# RSA by network
fig = plot_network_summary(all_rsa, title='RSA by network', ylabel='RSA r')
plt.show()

In [ ]:
# Top significant parcels — RSA
for run_type in cfg.run_types:
    fig = plot_isc_parcels(all_rsa[run_type], run_type=run_type, top_n=20)
    plt.show()

## 4. Brain-Behavior RSA (subject × subject)

In [ ]:
# Load political affiliation scores
# camp_support: 0 = far right, 100 = far left
# load behavioral data frame and merge with subject data frame to combine political attitude scores with BIDS IDs
behavioral_df = pd.read_csv("/path/to/behavioral_analyses/data/political_attitude_q_08122025.csv")
behavioral_df["subject_code"] = behavioral_df["subject_code"].str.replace(
    r'YY_PL_0*(\d+)', r'YY_PL_\1', regex=True)
subject_df = pd.read_csv("/path/to/behavioral_analyses/data/behavioral_with_bids.csv")

merged_df = pd.merge(behavioral_df, subject_df, on="subject_code", how="inner")

# Fix the measure column in the DataFrame first
merged_df["camp_support"] = pd.to_numeric(merged_df["camp_support"], errors="coerce")

# Filter merged_df to only those subjects
merged_df_filtered = merged_df[merged_df["bids_id"].isin(cfg.subjects)]
save_path = "/path/to/behavioral_analyses/data/merged_behavioral_bids.csv"
merged_df_filtered.to_csv(save_path, index=False)
print(f'Merged behavioral and BIDS data saved to: {save_path}')


In [ ]:
affiliation = load_affiliation(
    Path('/path/to/behavioral_analyses/data/merged_behavioral_bids.csv'),
    subjects  = cfg.subjects,
    score_col = 'camp_support',
    id_col    = 'bids_id',
)

# Build behavioral similarity matrix
beh_sim = make_behavioral_rdm(affiliation, cfg.subjects)
print('Behavioral similarity matrix shape:', beh_sim.shape)

In [ ]:
# Compute neural subject×subject similarity matrix for each condition
# Shape: (n_subjects, n_subjects, n_parcels)
neural_sims = {}
for run_type in cfg.run_types:
    _, neural_sims[run_type] = compute_brain_behavior_rsa(
        patterns[run_type], beh_sim, cfg.subjects
    )
    print(f"{run_type}: neural_sim shape = {neural_sims[run_type].shape}")

In [ ]:
# Run brain-behavior RSA for each condition
bb_results = {}
bb_nulls   = {}

for run_type in cfg.run_types:
    print(f'\n--- {run_type} ---')
    obs_r, p_vals, null = permutation_test_brain_behavior(
        patterns[run_type], beh_sim, cfg.subjects, cfg)
    rejected, p_fdr = fdr_correct(p_vals, cfg.fdr_q)

    bb_results[run_type] = results_to_dataframe(
        parcel_names, obs_r, p_vals, rejected, p_fdr)
    bb_nulls[run_type]   = (obs_r, p_vals, null)
    print(f'  {rejected.sum()} / {len(parcel_names)} parcels significant | '
          f'mean r = {obs_r.mean():.3f}')

### Brain-Behavior Visualizations

In [ ]:
# Heatmaps: neural vs behavioral for a parcel of interest
pcc_idx = parcel_names.index(pcc_parcels[0])

for run_type in cfg.run_types:
    fig = plot_similarity_matrices(
        neural_sims[run_type], beh_sim, cfg.subjects,
        pcc_idx, parcel_name=pcc_parcels[0], run_type=run_type)
    plt.show()

    fig = plot_brain_behavior_scatter(
        neural_sims[run_type], beh_sim, cfg.subjects,
        pcc_idx, parcel_name=pcc_parcels[0], run_type=run_type)
    plt.show()

In [ ]:
# Bar chart: mean brain-behavior RSA r across conditions
fig = plot_brain_behavior_bar(bb_results,
        title='Brain–behavior RSA (political affiliation)')
plt.show()

In [ ]:
# Top significant parcels — brain-behavior RSA
for run_type in cfg.run_types:
    fig = plot_isc_parcels(bb_results[run_type], run_type=run_type,
                            top_n=20, color='#9467bd')
    plt.show()

In [ ]:
# For a parcel of interest: heatmaps of neural vs behavioral similarity
# and a scatter plot of subject pairs
run_type = 'AntiLeft'
_, neural_sim = compute_brain_behavior_rsa(
    patterns[run_type], beh_sim, cfg.subjects)

fig = plot_similarity_matrices(
    neural_sim, beh_sim, cfg.subjects, pcc_idx,
    parcel_name=pcc_parcels[0], run_type=run_type)
plt.show()

fig = plot_brain_behavior_scatter(
    neural_sim, beh_sim, cfg.subjects, pcc_idx,
    parcel_name=pcc_parcels[0], run_type=run_type)
plt.show()

In [ ]:
# Network-level brain-behavior RSA
fig = plot_network_summary(bb_results,
        title='Brain–behavior RSA by network',
        ylabel='Brain–behavior RSA r')
plt.show()